In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

# Variance Inflation Factor (VIF)

The **Variance Inflation Factor (VIF)** quantifies how much the variance of an estimated regression coefficient is inflated due to **linear dependence among predictors**.

## Setup

Suppose we have predictors:

$$
X_1, X_2, \dots, X_n
$$

and we are concerned about collinearity among them.

For each predictor $X_j$, we fit an **auxiliary regression**:

$$
\hat X_j = \sum_{\substack{k=1 \\ k \neq j}}^n \beta_k X_k
$$

That is, we try to explain $X_j$ using all of the other predictors.

## Definition

Let $R_j^2$ be the correlation between the predicted values $\hat X_j$ and the true values $X_j$.

The Variance Inflation Factor for $X_j$ is:

$$
\text{VIF}(X_j) = \frac{1}{1 - R_j^2}
$$

## Interpretation

- If $R_j^2 \approx 0$, then $\text{VIF}(X_j) \approx 1$:  
  the predictor is not explained by the others, so there is **no multicollinearity**.

- If $R_j^2$ is large, then $\text{VIF}(X_j)$ is much greater than 1:  
  the predictor can be well explained by the others, so its coefficient in a regression will have **inflated variance**.

- If $R_j^2 = 1$, then $\text{VIF}(X_j) = \infty$:  
  predictor $X_j$ is an **exact linear combination** of the others (perfect multicollinearity).

## Rule of Thumb

- $\text{VIF} < 5$: Generally acceptable.  
- $5 \leq \text{VIF} < 10$: Indicates moderate multicollinearity.  
- $\text{VIF} \geq 10$: Suggests serious multicollinearity issues.

# Create data

In [ ]:
import numpy as np
import pandas as pd

n = 1000

# Independent random variables
x1 = np.random.randn(n)
x2 = np.random.randn(n)
x3 = np.random.randn(n)
x4 = np.random.randn(n)
x5 = np.random.randn(n)
x6 = np.random.randn(n)
x7 = np.random.randn(n)
x8 = np.random.randn(n)
x9 = np.random.randn(n)

# Create a 10th variable as a linear combination of all others
x10 = np.random.randn(n)                                   # Uncorrelated with others
#x10 = (0.2*x1 + 0.2*x2 - 0.2*x3 + 0.2*x4 + 0.2*x5 
       #- 0.2*x6 + 0.2*x7 + 0.2*x8 - 0.2*x9 
       #+ 0.0*np.random.randn(n))                          # Correlated with all
x10 = (0.2*x1 + 0.1*x2 - 0.3*x3 + 0.25*x4 + 0.15*x5 
       - 0.2*x6 + 0.1*x7 + 0.05*x8 - 0.1*x9 + 0.1*np.random.randn(n))              # Perfectly collinear

In [41]:
# Put in DataFrame
df = pd.DataFrame({
    "x1": x1,
    "x2": x2,
    "x3": x3,
    "x4": x4,
    "x5": x5,
    "x6": x6,
    "x7": x7,
    "x8": x8,
    "x9": x9,
    "x10": x10
})

df.corr()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10
x1,1.000000,0.019682,0.000464,-0.008001,0.029313,-0.055285,-0.043823,0.031703,0.048040,0.367682
x2,0.019682,1.000000,0.038820,-0.044054,0.018164,0.014983,-0.011670,0.041607,-0.010135,0.167696
x3,0.000464,0.038820,1.000000,0.087429,0.081001,0.002699,-0.051018,0.003540,0.007842,-0.535025
x4,-0.008001,-0.044054,0.087429,1.000000,0.011735,0.083327,-0.006914,0.004963,-0.018942,0.380387
x5,0.029313,0.018164,0.081001,0.011735,1.000000,0.003072,0.039963,0.016303,0.002136,0.266875
x6,-0.055285,0.014983,0.002699,0.083327,0.003072,1.000000,-0.054057,0.031040,0.015289,-0.371470
x7,-0.043823,-0.011670,-0.051018,-0.006914,0.039963,-0.054057,1.000000,0.032013,0.022417,0.224451
x8,0.031703,0.041607,0.003540,0.004963,0.016303,0.031040,0.032013,1.000000,-0.002461,0.112501
x9,0.048040,-0.010135,0.007842,-0.018942,0.002136,0.015289,0.022417,-0.002461,1.000000,-0.180847
x10,0.367682,0.167696,-0.535025,0.380387,0.266875,-0.371470,0.224451,0.112501,-0.180847,1.000000


# Compute VIF

In [42]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Compute VIFs without adding a constant
vif = pd.Series(
    [variance_inflation_factor(df.values, i) 
     for i in range(df.shape[1])],
    index=df.columns
)

print("Variance Inflation Factors (no constant):")
print(vif)


Variance Inflation Factors (no constant):
x1      4.526667
x2      2.160225
x3     11.170056
x4      7.300215
x5      3.385301
x6      5.197856
x7      1.971883
x8      1.253163
x9      1.931505
x10    28.467089
dtype: float64
